# HDF5 CNN Predictions → NIfTI Orientation Mapping

Figure out the transformation needed to convert HDF5 CNN-corrected velocity data
into NIfTI format that can be fed into the existing DICOM writing pipeline.

**Strategy**: Compare the HDF5 data against the existing uncorrected velocity NIfTIs
(padded FOV) — which have the correct NIfTI axis convention and affine — to determine
the axis permutation, flips, and sign needed.

In [1]:
import platform
import numpy as np
import pandas as pd
import h5py
import nibabel as nib
from pathlib import Path
from itertools import permutations

from vascular_superenhancement.utils.path_config import load_path_config, _PROJECT_ROOT

config_name = "local_mac" if platform.system() == "Darwin" else "all_patients"
pc = load_path_config(config_name)

PATIENT_DATA_DIR = pc.working_dir / "patient_data"
HDF5_PATH = _PROJECT_ROOT / "working_dir" / "cardiac_processed_cnn_corrected.hdf5"

Found project root at: /home/ayeluru/vascular-superenhancement-4d-flow


## Step 1: Load uncorrected vz NIfTI (padded FOV) for Balboloop (I_to_S)

In [2]:
pid = "Balboloop"
t_idx = 0

nii_path = (
    PATIENT_DATA_DIR / pid / "nifti"
    / f"4d_flow_vz_{pid}_per_timepoint_full_fov"
    / f"4d_flow_vz_{pid}_frame_{t_idx:02d}.nii.gz"
)
nii = nib.load(str(nii_path))
nii_data = nii.get_fdata(dtype=np.float32)
nii_affine = nii.affine

print(f"Patient: {pid} (I_to_S)")
print(f"NIfTI path: {nii_path}")
print(f"NIfTI shape: {nii_data.shape}")
print(f"NIfTI affine:\n{nii_affine}")

Patient: Balboloop (I_to_S)
NIfTI path: /home/ayeluru/vascular-superenhancement-4d-flow/working_dir/all_patients/patient_data/Balboloop/nifti/4d_flow_vz_Balboloop_per_timepoint_full_fov/4d_flow_vz_Balboloop_frame_00.nii.gz
NIfTI shape: (256, 256, 140)
NIfTI affine:
[[ -1.40625083   0.           0.         158.86599731]
 [  0.          -1.40625083   0.         148.21499634]
 [  0.           0.           1.79987395 -89.54589844]
 [  0.           0.           0.           1.        ]]


## Step 2: Load HDF5 vz for the same patient

In [3]:
with h5py.File(HDF5_PATH, "r") as f:
    hdf5_full = f[pid][:]  # (D0, D1, D2, T, 3)

print(f"HDF5 full shape: {hdf5_full.shape}")

# Extract vz at t=0: component index 2
hdf5_vz = hdf5_full[:, :, :, t_idx, 2].astype(np.float32)
print(f"HDF5 vz volume shape: {hdf5_vz.shape}")
print(f"NIfTI vz volume shape: {nii_data.shape}")

HDF5 full shape: (256, 256, 140, 20, 3)
HDF5 vz volume shape: (256, 256, 140)
NIfTI vz volume shape: (256, 256, 140)


## Step 3: Find the axis mapping — brute-force all permutations and flips

Try all 48 combinations (6 permutations × 8 flip combos) of the HDF5 axes
and correlate against the NIfTI at a mid-z slice. The best match tells us
the correct transformation.

In [4]:
nii_shape = nii_data.shape  # (I, J, K)
hdf5_shape = hdf5_vz.shape  # (D0, D1, D2)

print(f"NIfTI shape: {nii_shape}")
print(f"HDF5 shape:  {hdf5_shape}")

z_mid = nii_shape[2] // 2
nii_slice = nii_data[:, :, z_mid]  # (I, J) — ground truth

results = []

for perm in permutations([0, 1, 2]):
    transposed = np.transpose(hdf5_vz, perm)
    # After transposing, check if the z-dimension (axis 2) matches
    if transposed.shape[2] != nii_shape[2]:
        continue
    # Also check that the 2D slice dims match
    if transposed.shape[0] != nii_shape[0] or transposed.shape[1] != nii_shape[1]:
        continue

    for flip_0 in [False, True]:
        for flip_1 in [False, True]:
            for flip_2 in [False, True]:
                vol = transposed.copy()
                if flip_0:
                    vol = vol[::-1, :, :]
                if flip_1:
                    vol = vol[:, ::-1, :]
                if flip_2:
                    vol = vol[:, :, ::-1]

                hdf5_slice = vol[:, :, z_mid]

                mask = (np.abs(nii_slice) > 1) & (np.abs(hdf5_slice) > 1)
                if mask.sum() < 100:
                    continue

                corr = np.corrcoef(nii_slice[mask], hdf5_slice[mask])[0, 1]
                results.append({
                    "perm": perm,
                    "flip_0": flip_0,
                    "flip_1": flip_1,
                    "flip_2": flip_2,
                    "corr": corr,
                    "shape": vol.shape,
                })

results_df = pd.DataFrame(results).sort_values("corr", key=abs, ascending=False)
print(f"\nTop 10 transformations (by |correlation|):")
print(results_df.head(10).to_string(index=False))

NIfTI shape: (256, 256, 140)
HDF5 shape:  (256, 256, 140)

Top 10 transformations (by |correlation|):
     perm  flip_0  flip_1  flip_2      corr           shape
(1, 0, 2)   False   False   False  0.996696 (256, 256, 140)
(1, 0, 2)   False   False    True  0.279551 (256, 256, 140)
(0, 1, 2)    True    True   False  0.018913 (256, 256, 140)
(0, 1, 2)    True    True    True  0.018056 (256, 256, 140)
(1, 0, 2)    True   False    True  0.011524 (256, 256, 140)
(0, 1, 2)   False   False   False  0.005870 (256, 256, 140)
(1, 0, 2)    True   False   False  0.004469 (256, 256, 140)
(0, 1, 2)   False   False    True  0.004083 (256, 256, 140)
(1, 0, 2)    True    True   False -0.003857 (256, 256, 140)
(0, 1, 2)   False    True   False -0.003331 (256, 256, 140)


## Step 3b: Visualize the best match

In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

best = results_df.iloc[0]
print(f"Best transformation:")
print(f"  Permutation: {best['perm']}")
print(f"  Flip axes: 0={best['flip_0']}, 1={best['flip_1']}, 2={best['flip_2']}")
print(f"  Correlation: {best['corr']:.6f}")
print(f"  Needs vz negation: {best['corr'] < 0}")

# Apply best transformation
vol = np.transpose(hdf5_vz, best["perm"])
if best["flip_0"]:
    vol = vol[::-1, :, :]
if best["flip_1"]:
    vol = vol[:, ::-1, :]
if best["flip_2"]:
    vol = vol[:, :, ::-1]
if best["corr"] < 0:
    vol = -vol

out_dir = Path("hdf5_orientation_check")
out_dir.mkdir(exist_ok=True)

z_slices = [nii_shape[2] // 4, nii_shape[2] // 2, 3 * nii_shape[2] // 4]
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
vmax = 200

for row, z in enumerate(z_slices):
    nii_sl = nii_data[:, :, z]
    hdf5_sl = vol[:, :, z]
    diff_sl = hdf5_sl - nii_sl

    axes[row, 0].imshow(nii_sl.T, cmap="RdBu_r", origin="lower", vmin=-vmax, vmax=vmax)
    axes[row, 0].set_title(f"NIfTI vz (z={z})")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(hdf5_sl.T, cmap="RdBu_r", origin="lower", vmin=-vmax, vmax=vmax)
    axes[row, 1].set_title(f"HDF5 transformed vz (z={z})")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(diff_sl.T, cmap="RdBu_r", origin="lower", vmin=-vmax, vmax=vmax)
    axes[row, 2].set_title(f"Diff (z={z})")
    axes[row, 2].axis("off")

fig.suptitle(
    f"{pid} (I_to_S) — Best transform: perm={best['perm']}, "
    f"flip=({best['flip_0']},{best['flip_1']},{best['flip_2']}), "
    f"negate={best['corr'] < 0}, corr={best['corr']:.4f}",
    fontsize=12,
)
fig.tight_layout()
fig.savefig(out_dir / f"{pid}_best_match.png", dpi=120)
plt.close(fig)
print(f"Saved to {out_dir / f'{pid}_best_match.png'}")

Best transformation:
  Permutation: (1, 0, 2)
  Flip axes: 0=False, 1=False, 2=False
  Correlation: 0.996696
  Needs vz negation: False
Saved to hdf5_orientation_check/Balboloop_best_match.png


## Step 4: Repeat for Biswifo (S_to_I) — verify same transform works

In [6]:
pid2 = "Biswifo"

# Load NIfTI
nii_path2 = (
    PATIENT_DATA_DIR / pid2 / "nifti"
    / f"4d_flow_vz_{pid2}_per_timepoint_full_fov"
    / f"4d_flow_vz_{pid2}_frame_{t_idx:02d}.nii.gz"
)
nii2 = nib.load(str(nii_path2))
nii_data2 = nii2.get_fdata(dtype=np.float32)
print(f"\nPatient: {pid2} (S_to_I)")
print(f"NIfTI shape: {nii_data2.shape}")
print(f"NIfTI affine:\n{nii2.affine}")

# Load HDF5
with h5py.File(HDF5_PATH, "r") as f:
    hdf5_full2 = f[pid2][:]
hdf5_vz2 = hdf5_full2[:, :, :, t_idx, 2].astype(np.float32)
print(f"HDF5 shape: {hdf5_full2.shape}")
print(f"HDF5 vz shape: {hdf5_vz2.shape}")

# Run same brute-force search
nii_shape2 = nii_data2.shape
z_mid2 = nii_shape2[2] // 2
nii_slice2 = nii_data2[:, :, z_mid2]

results2 = []
for perm in permutations([0, 1, 2]):
    transposed = np.transpose(hdf5_vz2, perm)
    if transposed.shape[2] != nii_shape2[2]:
        continue
    if transposed.shape[0] != nii_shape2[0] or transposed.shape[1] != nii_shape2[1]:
        continue

    for flip_0 in [False, True]:
        for flip_1 in [False, True]:
            for flip_2 in [False, True]:
                vol2 = transposed.copy()
                if flip_0:
                    vol2 = vol2[::-1, :, :]
                if flip_1:
                    vol2 = vol2[:, ::-1, :]
                if flip_2:
                    vol2 = vol2[:, :, ::-1]

                hdf5_slice2 = vol2[:, :, z_mid2]
                mask2 = (np.abs(nii_slice2) > 1) & (np.abs(hdf5_slice2) > 1)
                if mask2.sum() < 100:
                    continue

                corr2 = np.corrcoef(nii_slice2[mask2], hdf5_slice2[mask2])[0, 1]
                results2.append({
                    "perm": perm,
                    "flip_0": flip_0,
                    "flip_1": flip_1,
                    "flip_2": flip_2,
                    "corr": corr2,
                    "shape": vol2.shape,
                })

results2_df = pd.DataFrame(results2).sort_values("corr", key=abs, ascending=False)
print(f"\nTop 10 transformations (by |correlation|):")
print(results2_df.head(10).to_string(index=False))


Patient: Biswifo (S_to_I)
NIfTI shape: (256, 256, 120)
NIfTI affine:
[[  -1.48437476    0.            0.          163.83900452]
 [   0.           -1.48437476    0.          177.16999817]
 [   0.            0.            1.59994555 -211.8809967 ]
 [   0.            0.            0.            1.        ]]
HDF5 shape: (256, 256, 120, 20, 3)
HDF5 vz shape: (256, 256, 120)

Top 10 transformations (by |correlation|):
     perm  flip_0  flip_1  flip_2      corr           shape
(1, 0, 2)   False   False    True  0.948641 (256, 256, 120)
(1, 0, 2)   False   False   False  0.323515 (256, 256, 120)
(1, 0, 2)   False    True   False  0.024715 (256, 256, 120)
(1, 0, 2)    True   False   False -0.021237 (256, 256, 120)
(1, 0, 2)   False    True    True  0.020453 (256, 256, 120)
(1, 0, 2)    True    True   False  0.017993 (256, 256, 120)
(0, 1, 2)    True   False    True  0.014068 (256, 256, 120)
(1, 0, 2)    True   False    True -0.013984 (256, 256, 120)
(1, 0, 2)    True    True    True  0.011794

In [7]:
best2 = results2_df.iloc[0]
print(f"Best transformation for {pid2}:")
print(f"  Permutation: {best2['perm']}")
print(f"  Flip axes: 0={best2['flip_0']}, 1={best2['flip_1']}, 2={best2['flip_2']}")
print(f"  Correlation: {best2['corr']:.6f}")
print(f"  Needs vz negation: {best2['corr'] < 0}")

# Apply best transformation
vol2 = np.transpose(hdf5_vz2, best2["perm"])
if best2["flip_0"]:
    vol2 = vol2[::-1, :, :]
if best2["flip_1"]:
    vol2 = vol2[:, ::-1, :]
if best2["flip_2"]:
    vol2 = vol2[:, :, ::-1]
if best2["corr"] < 0:
    vol2 = -vol2

z_slices2 = [nii_shape2[2] // 4, nii_shape2[2] // 2, 3 * nii_shape2[2] // 4]
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for row, z in enumerate(z_slices2):
    nii_sl = nii_data2[:, :, z]
    hdf5_sl = vol2[:, :, z]
    diff_sl = hdf5_sl - nii_sl

    axes[row, 0].imshow(nii_sl.T, cmap="RdBu_r", origin="lower", vmin=-vmax, vmax=vmax)
    axes[row, 0].set_title(f"NIfTI vz (z={z})")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(hdf5_sl.T, cmap="RdBu_r", origin="lower", vmin=-vmax, vmax=vmax)
    axes[row, 1].set_title(f"HDF5 transformed vz (z={z})")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(diff_sl.T, cmap="RdBu_r", origin="lower", vmin=-vmax, vmax=vmax)
    axes[row, 2].set_title(f"Diff (z={z})")
    axes[row, 2].axis("off")

fig.suptitle(
    f"{pid2} (S_to_I) — Best transform: perm={best2['perm']}, "
    f"flip=({best2['flip_0']},{best2['flip_1']},{best2['flip_2']}), "
    f"negate={best2['corr'] < 0}, corr={best2['corr']:.4f}",
    fontsize=12,
)
fig.tight_layout()
fig.savefig(out_dir / f"{pid2}_best_match.png", dpi=120)
plt.close(fig)
print(f"Saved to {out_dir / f'{pid2}_best_match.png'}")

Best transformation for Biswifo:
  Permutation: (1, 0, 2)
  Flip axes: 0=False, 1=False, 2=True
  Correlation: 0.948641
  Needs vz negation: False
Saved to hdf5_orientation_check/Biswifo_best_match.png


## Step 5: Compare results — do I_to_S and S_to_I need different transforms?

In [8]:
print("=" * 60)
print(f"Balboloop (I_to_S):")
print(f"  perm={best['perm']}, flip=({best['flip_0']},{best['flip_1']},{best['flip_2']}), negate={best['corr'] < 0}")
print(f"  corr={best['corr']:.6f}")
print()
print(f"Biswifo (S_to_I):")
print(f"  perm={best2['perm']}, flip=({best2['flip_0']},{best2['flip_1']},{best2['flip_2']}), negate={best2['corr'] < 0}")
print(f"  corr={best2['corr']:.6f}")
print("=" * 60)

if best['perm'] == best2['perm']:
    print("\nSame permutation for both — good.")
else:
    print("\nDIFFERENT permutations — need per-patient handling.")

flips_match = (
    best['flip_0'] == best2['flip_0'] and
    best['flip_1'] == best2['flip_1'] and
    best['flip_2'] == best2['flip_2']
)
if flips_match:
    print("Same flips for both — good.")
else:
    print("DIFFERENT flips — likely z-flip differs by acquisition direction.")

if (best['corr'] < 0) == (best2['corr'] < 0):
    print("Same negation for both — good.")
else:
    print("DIFFERENT negation — vz sign differs by acquisition direction.")

Balboloop (I_to_S):
  perm=(1, 0, 2), flip=(False,False,False), negate=False
  corr=0.996696

Biswifo (S_to_I):
  perm=(1, 0, 2), flip=(False,False,True), negate=False
  corr=0.948641

Same permutation for both — good.
DIFFERENT flips — likely z-flip differs by acquisition direction.
Same negation for both — good.


## Step 6: Convert HDF5 → per-timepoint NIfTIs for all test patients

For each patient:
1. Transpose axes (1,0,2) on the spatial dims
2. Flip z for S_to_I patients
3. Save each velocity component at each timepoint as a NIfTI with the padded FOV affine

In [9]:
PILOT_PATIENTS = ["Balboloop", "Biswifo"]

VZ_FLOW_TAG = 5

def get_orig_direction(pid):
    catalog_path = PATIENT_DATA_DIR / pid / f"dicom_catalog_{pid}.csv"
    if not catalog_path.exists():
        return "UNKNOWN"
    catalog = pd.read_csv(catalog_path)
    vz_cat = catalog[catalog["tag_0x0043_0x1030"] == VZ_FLOW_TAG].copy()
    if len(vz_cat) == 0:
        return "UNKNOWN"
    vz_cat["time_index"] = (vz_cat["instancenumber"] - 1) % vz_cat["cardiacnumberofimages"]
    vz_cat["slice_index"] = (vz_cat["instancenumber"] - 1) // vz_cat["cardiacnumberofimages"]
    vz_cat["z"] = vz_cat["imagepositionpatient"].apply(lambda x: np.array(eval(x))[2])
    t0 = vz_cat[vz_cat["time_index"] == 0].sort_values("slice_index")
    z_diff = np.diff(t0["z"].values)
    if np.sum(z_diff > 0) > np.sum(z_diff < 0):
        return "I_to_S"
    elif np.sum(z_diff < 0) > np.sum(z_diff > 0):
        return "S_to_I"
    return "AMBIGUOUS"

COMP_NAMES = ["vx", "vy", "vz"]
OUTPUT_ROOT = _PROJECT_ROOT / "working_dir" / "cnn_corrected_niftis"

with h5py.File(HDF5_PATH, "r") as f:
    for pi, pid in enumerate(PILOT_PATIENTS):
        orig_dir = get_orig_direction(pid)
        flip_z = orig_dir == "S_to_I"
        print(f"[{pi+1}/{len(PILOT_PATIENTS)}] {pid} ({orig_dir}){' [flip z]' if flip_z else ''}...", flush=True)

        if pid not in f:
            print(f"  NOT in HDF5, skipping")
            continue

        # Get padded FOV affine from existing uncorrected vx NIfTI
        ref_nii_path = (
            PATIENT_DATA_DIR / pid / "nifti"
            / f"4d_flow_vx_{pid}_per_timepoint_full_fov"
            / f"4d_flow_vx_{pid}_frame_00.nii.gz"
        )
        if not ref_nii_path.exists():
            print(f"  Reference NIfTI not found, skipping")
            continue
        ref_affine = nib.load(str(ref_nii_path)).affine

        # Load HDF5: (D0, D1, Z, T, 3)
        hdf5_data = f[pid][:].astype(np.float32)

        # Step 1: Transpose spatial axes → (D1, D0, Z, T, 3) to match NIfTI (I, J, K, T, C)
        hdf5_data = np.transpose(hdf5_data, (1, 0, 2, 3, 4))

        # Step 2: Flip z for S_to_I patients
        if flip_z:
            hdf5_data = hdf5_data[:, :, ::-1, :, :]

        n_t = hdf5_data.shape[3]
        pid_out_dir = OUTPUT_ROOT / pid
        pid_out_dir.mkdir(parents=True, exist_ok=True)

        # Step 3: Save per-component, per-timepoint NIfTIs
        for c_idx, comp in enumerate(COMP_NAMES):
            comp_dir = pid_out_dir / f"4d_flow_{comp}_cnn_corr"
            comp_dir.mkdir(exist_ok=True)

            for t in range(n_t):
                vol = hdf5_data[:, :, :, t, c_idx]
                nii = nib.Nifti1Image(vol, ref_affine)
                out_path = comp_dir / f"4d_flow_{comp}_cnn_corr_{pid}_frame_{t:02d}.nii.gz"
                nib.save(nii, out_path)

        print(f"  Saved {n_t} timepoints × 3 components to {pid_out_dir}")

print(f"\nDone. All NIfTIs in: {OUTPUT_ROOT}")

[1/2] Balboloop (I_to_S)...
  Saved 20 timepoints × 3 components to /home/ayeluru/vascular-superenhancement-4d-flow/working_dir/cnn_corrected_niftis/Balboloop
[2/2] Biswifo (S_to_I) [flip z]...
  Saved 20 timepoints × 3 components to /home/ayeluru/vascular-superenhancement-4d-flow/working_dir/cnn_corrected_niftis/Biswifo

Done. All NIfTIs in: /home/ayeluru/vascular-superenhancement-4d-flow/working_dir/cnn_corrected_niftis


## Step 7: Write DICOMs using CNN-corrected velocity NIfTIs + blend_50 mag

In [10]:
import zipfile
from pydicom.uid import generate_uid
from vascular_superenhancement.data_management.patients import Patient
from vascular_superenhancement.data_management.nifti_to_dicom import NiftiToDicomConverter

INFERENCE_ROOT = _PROJECT_ROOT / "working_dir" / "inference" / "glowing-microwave_epoch-69"
MAG_METHOD = "blend_50"
DICOM_OUTPUT_ROOT = _PROJECT_ROOT / "working_dir" / "cnn_corrected_dicoms"

for pid in PILOT_PATIENTS:
    print(f"\n{'='*60}")
    print(f"Patient: {pid}")
    print(f"{'='*60}")

    patient = Patient(path_config=pc, phonetic_id=pid, debug=False, config="all_patients")
    converter = NiftiToDicomConverter.from_patient(patient)
    num_tp = patient.num_timepoints

    mag_dir = INFERENCE_ROOT / pid / "predicted_mag" / MAG_METHOD
    vel_dir = OUTPUT_ROOT / pid

    study_uid = generate_uid()
    series_uids = {2: generate_uid(), 3: generate_uid(),
                   4: generate_uid(), 5: generate_uid()}

    print(f"  Mag dir: {mag_dir}")
    print(f"  Vel dir: {vel_dir}")
    print(f"  Writing {num_tp} timepoints...")

    for t in range(num_tp):
        mag_path = mag_dir / f"pred_mag_{pid}_frame_{t:02d}.nii.gz"
        assert mag_path.exists(), f"Missing mag: {mag_path}"

        vel_paths = {
            "vx": vel_dir / "4d_flow_vx_cnn_corr" / f"4d_flow_vx_cnn_corr_{pid}_frame_{t:02d}.nii.gz",
            "vy": vel_dir / "4d_flow_vy_cnn_corr" / f"4d_flow_vy_cnn_corr_{pid}_frame_{t:02d}.nii.gz",
            "vz": vel_dir / "4d_flow_vz_cnn_corr" / f"4d_flow_vz_cnn_corr_{pid}_frame_{t:02d}.nii.gz",
        }
        for k, p in vel_paths.items():
            assert p.exists(), f"Missing {k}: {p}"

        out_dir = DICOM_OUTPUT_ROOT / pid / "cnn_corrected"

        converter.write_timepoint_with_velocities_to_dicoms(
            mag_prediction_path=mag_path,
            velocity_paths=vel_paths,
            output_dir=out_dir,
            timepoint=t,
            study_uid=study_uid,
            series_uids=series_uids,
            overwrite=True,
            series_descriptions={
                2: f"CNN Mag ({MAG_METHOD})",
                3: "CNN Corrected Vx",
                4: "CNN Corrected Vy",
                5: "CNN Corrected Vz",
            },
        )
        print(f"  t={t}: done", flush=True)

    # Zip the DICOMs
    dicom_dir = DICOM_OUTPUT_ROOT / pid / "cnn_corrected"
    zip_path = DICOM_OUTPUT_ROOT / pid / f"{pid}_cnn_corrected.zip"
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for fp in sorted(dicom_dir.rglob("*")):
            if fp.is_file():
                zf.write(fp, arcname=fp.relative_to(dicom_dir))
    print(f"  Zipped to {zip_path}")

print(f"\nAll done. DICOMs in: {DICOM_OUTPUT_ROOT}")


Patient: Balboloop


2026-03-11 19:21:21,839 - INFO - Successfully loaded existing 4D Flow catalog for patient Balboloop


  Mag dir: /home/ayeluru/vascular-superenhancement-4d-flow/working_dir/inference/glowing-microwave_epoch-69/Balboloop/predicted_mag/blend_50
  Vel dir: /home/ayeluru/vascular-superenhancement-4d-flow/working_dir/cnn_corrected_niftis/Balboloop
  Writing 20 timepoints...


2026-03-11 19:21:35,737 - INFO - Using fixed DICOM intensity range: 0.0, 3473.0
2026-03-11 19:21:35,807 - INFO - Prediction intensity range: 0.0, 0.5143117702007332


  t=0: done


...

  t=18: done


2026-03-11 19:34:08,595 - INFO - Processed timepoint 18: 120 slices (mag + velocity)
2026-03-11 19:34:11,995 - INFO - Using fixed DICOM intensity range: 0.0, 3746.0
2026-03-11 19:34:12,036 - INFO - Prediction intensity range: 0.0, 0.45739983376861326


  t=19: done


2026-03-11 19:34:21,943 - INFO - Processed timepoint 19: 120 slices (mag + velocity)


  Zipped to /home/ayeluru/vascular-superenhancement-4d-flow/working_dir/cnn_corrected_dicoms/Biswifo/Biswifo_cnn_corrected.zip

All done. DICOMs in: /home/ayeluru/vascular-superenhancement-4d-flow/working_dir/cnn_corrected_dicoms
